# M9.11 · Bölütleme — nokta/çizgi, k-means, GMM, SLIC süper pikseller

Her piksele bir etiket: Laplacian ile tekil nokta ve yönlü çizgi tespiti, kümelemeyle bölge bölütleme (k-means, Gauss karışım modeli ve EM) ve SLIC süper pikselleri. Elle bir k-means koşusu, SLIC uzaklığı ve EM sorumluluğu; saf Python'da k-means'i scikit-learn ile birebir; NumPy'da vektörel k-means ve renk nicemleme; PyTorch'ta tam kovaryanslı EM'i scikit-learn'ün `GaussianMixture`'ı ile aynı sayılara getiriyorsun; en sonda SLIC'i tensör işlemleriyle yazıp pencereli döngü sürümüyle piksel piksel aynı etiketleri üretiyorsun. Ek alıştırmalar Otsu ve uyarlamalı eşikleme, süper piksel + k-means (alıştırma 6.1) ve bölge büyütme.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m9-11-segmentation.html](https://m-gonul.github.io/pytorch-lab/m9-11-segmentation.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m9_11_segmentation.py
CHECKS = {
 "b": "import numpy as np\nfrom scipy import ndimage\nfrom skimage import data\nfrom sklearn.cluster import KMeans\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"cv2\", \"scipy\", \"sklearn\"}))\n\n    def pt():\n        rng = np.random.default_rng(0)\n        img = (rng.random((30, 40)) * 20).astype(int); img[10, 12] = 200; img[20, 30] = 0; img[5, 5] = 220\n        got = np.array(need(ns, \"point_detect\")(img.tolist(), 900))\n        R_ = ndimage.correlate(img, np.array([[1, 1, 1], [1, -8, 1], [1, 1, 1]]), mode=\"constant\")\n        want = (np.abs(R_) >= 900).astype(int); want[0] = want[-1] = 0; want[:, 0] = want[:, -1] = 0\n        ensure(np.array_equal(got, want), \"nokta tespiti\")\n        return f\"bulunan: {list(zip(*np.nonzero(got)))}\"\n    R.test(\"point_detect\", pt)\n\n    def ln():\n        img = np.zeros((25, 25), int); img[12, :] = 100; np.fill_diagonal(img, 100)\n        ks = {\"h\": [[-1, -1, -1], [2, 2, 2], [-1, -1, -1]], \"v\": [[-1, 2, -1]] * 3, \"d1\": [[2, -1, -1], [-1, 2, -1], [-1, -1, 2]], \"d2\": [[-1, -1, 2], [-1, 2, -1], [2, -1, -1]]}\n        rows = []\n        for kind, k in ks.items():\n            got = np.array(need(ns, \"line_detect\")(img.tolist(), kind, 300))\n            want = (ndimage.correlate(img, np.array(k), mode=\"constant\") >= 300).astype(int); want[0] = want[-1] = 0; want[:, 0] = want[:, -1] = 0\n            ensure(np.array_equal(got, want), f\"{kind}\")\n            rows.append([kind, int(got.sum())])\n        ensure(rows[0][1] > 15 and rows[2][1] > 15 and rows[1][1] == 0 and rows[3][1] <= 1, f\"yatay ve d1 çizgiler bulunmalı: {rows}\")\n        R.table(\"Yatay + ana köşegen çizgili görüntü, T = 300\", [\"çekirdek\", \"piksel\"], rows)\n    R.test(\"line_detect (yalnız pozitif yanıt)\", ln)\n\n    def km():\n        rng = np.random.default_rng(1)\n        Z = np.concatenate([rng.normal(c, 1.0, (40, 2)) for c in ([0, 0], [6, 1], [2, 7])])\n        init = Z[[0, 1, 2]]\n        lab, m = need(ns, \"kmeans\")(Z.tolist(), init.tolist())\n        sk = KMeans(3, init=init, n_init=1, tol=0, algorithm=\"lloyd\").fit(Z)\n        ensure(list(lab) == sk.labels_.tolist(), \"etiketler sklearn ile farklı\")\n        close(np.array(m), sk.cluster_centers_, atol=1e-9, what=\"merkezler\")\n        ensure(ns[\"kmeans\"]([[1], [2], [3], [10], [11], [12]], [[1], [2]])[1] == [[2.0], [11.0]], \"elle örnek\")\n    R.test(\"kmeans = sklearn KMeans (lloyd)\", km)\n",
 "c": "import numpy as np\nfrom sklearn.cluster import KMeans\nfrom skimage import data\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"yalnız iterasyon döngüsü\", lambda: max_loops(src, 1, func=\"kmeans_np\"))\n    img = data.astronaut()[::4, ::4]\n    Z = img.reshape(-1, 3).astype(np.float64)\n\n    for k, seed in [(3, 0), (6, 1), (12, 2)]:\n        def t(k=k, seed=seed):\n            init = Z[np.random.default_rng(seed).choice(len(Z), k, replace=False)]\n            lab, m = need(ns, \"kmeans_np\")(Z, init)\n            sk = KMeans(k, init=init, n_init=1, tol=0, algorithm=\"lloyd\", max_iter=300).fit(Z)\n            ensure(np.array_equal(lab, sk.labels_), f\"{(lab != sk.labels_).sum()} etiket farklı\")\n            close(m, sk.cluster_centers_, atol=1e-9, what=\"merkezler\")\n        R.test(f\"kmeans_np = sklearn (k = {k})\", t)\n\n    def q():\n        rows = []\n        prev = np.inf\n        for k in (2, 4, 8, 16):\n            out, lab = need(ns, \"quantize\")(img, k)\n            ensure(out.dtype == np.uint8 and out.shape == img.shape and lab.shape == img.shape[:2], \"şekiller\")\n            ensure(len(np.unique(out.reshape(-1, 3), axis=0)) <= k, \"en fazla k renk\")\n            mse = float(((out.astype(float) - img) ** 2).mean())\n            rows.append([k, f\"{mse:.1f}\", f\"{10 * np.log10(255 ** 2 / mse):.2f}\"])\n            prev = mse\n        ensure(float(rows[-1][1]) < float(rows[0][1]), \"k arttıkça hata azalmalı\")\n        R.table(\"Renk nicemleme (astronaut 128²)\", [\"k\", \"MSE\", \"PSNR (dB)\"], rows)\n    R.test(\"quantize\", q)\n",
 "d": "import warnings\nimport numpy as np\nimport torch\nfrom sklearn.mixture import GaussianMixture\nfrom sklearn.cluster import KMeans\nfrom sklearn.metrics import adjusted_rand_score\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    warnings.filterwarnings(\"ignore\")\n    Z = data.astronaut()[::4, ::4].reshape(-1, 3).astype(np.float64) / 255\n    rng = np.random.default_rng(0)\n    mu0 = Z[rng.choice(len(Z), 5, replace=False)]\n\n    def sk():\n        mu, cov, w, ll = need(ns, \"gmm_em\")(torch.tensor(Z), torch.tensor(mu0), 30)\n        c0 = np.cov(Z.T) + 1e-6 * np.eye(3)\n        g = GaussianMixture(5, covariance_type=\"full\", weights_init=np.full(5, 0.2), means_init=mu0,\n                            precisions_init=np.stack([np.linalg.inv(c0)] * 5), max_iter=30, tol=0, reg_covar=1e-6).fit(Z)\n        close(mu.numpy(), g.means_, atol=1e-8, what=\"ortalamalar\")\n        close(cov.numpy(), g.covariances_, atol=1e-8, what=\"kovaryanslar\")\n        close(w.numpy(), g.weights_, atol=1e-8, what=\"ağırlıklar\")\n        close(float(ll), g.score(Z), atol=1e-8, what=\"ortalama log-olabilirlik\")\n        return f\"30 iterasyon, log-olabilirlik {float(ll):.4f}\"\n    R.test(\"gmm_em = sklearn GaussianMixture\", sk)\n\n    def ell():\n        g = np.random.default_rng(3)\n        A = np.array([[3.0, 0], [0, 0.3]])\n        rot = lambda t: np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])\n        X = np.concatenate([g.normal(size=(300, 2)) @ (rot(0.6) @ A).T, g.normal(size=(300, 2)) @ (rot(0.6) @ A).T + [0, 2.2]])\n        y = np.repeat([0, 1], 300)\n        init = X[[0, 300]]\n        km = KMeans(2, init=init, n_init=1).fit(X).labels_\n        mu, cov, w, _ = ns[\"gmm_em\"](torch.tensor(X), torch.tensor(init), 100)\n        from scipy.stats import multivariate_normal as mvn\n        lab = np.argmax(np.stack([w[k].item() * mvn(mu[k].numpy(), cov[k].numpy()).pdf(X) for k in range(2)], 1), 1)\n        a_km, a_g = adjusted_rand_score(y, km), adjusted_rand_score(y, lab)\n        ensure(a_g > a_km + 0.2, f\"GMM ARI {a_g:.2f}, k-means ARI {a_km:.2f}\")\n        R.table(\"İki paralel, uzamış küme\", [\"yöntem\", \"ARI\"], [[\"k-means (küresel)\", f\"{a_km:.2f}\"], [\"GMM (tam kovaryans)\", f\"{a_g:.2f}\"]],\n                note=\"k-means kümeleri ortalamaya Öklid uzaklığıyla ayırır; eliptik ve yakın kümelerde sınırı yanlış yere çizer.\")\n    R.test(\"eliptik kümelerde GMM > k-means\", ell)\n",
 "e": "import math\nimport numpy as np\nfrom skimage import data, color\nfrom labcheck import need, ensure, bench, fmt_time\n\n\ndef init_centers(L, S):\n    H, W, _ = L.shape\n    P = np.pad(L, ((1, 1), (1, 1), (0, 0)), mode=\"edge\")\n    G = ((P[1:-1, 2:] - P[1:-1, :-2]) ** 2).sum(-1) + ((P[2:, 1:-1] - P[:-2, 1:-1]) ** 2).sum(-1)\n    cs = []\n    for y in range(S // 2, H, S):\n        for x in range(S // 2, W, S):\n            best = None\n            for dy in (-1, 0, 1):\n                for dx in (-1, 0, 1):\n                    yy, xx = y + dy, x + dx\n                    if 0 <= yy < H and 0 <= xx < W and (best is None or G[yy, xx] < best[0]):\n                        best = (G[yy, xx], yy, xx)\n            cs.append([*L[best[1], best[2]], best[1], best[2]])\n    return np.array(cs, np.float64)\n\n\ndef slic_ref(L, K, c=10.0, iters=10):\n    H, W, _ = L.shape\n    S = int(round(math.sqrt(H * W / K)))\n    C = init_centers(L, S)\n    lab = -np.ones((H, W), np.int64)\n    yy, xx = np.mgrid[:H, :W]\n    F = np.concatenate([L, yy[..., None], xx[..., None]], -1).reshape(-1, 5)\n    for _ in range(iters):\n        d = np.full((H, W), np.inf)\n        for k, (l, a, b, cy, cx) in enumerate(C):\n            y0, y1 = max(0, math.ceil(cy - S)), min(H - 1, math.floor(cy + S))\n            x0, x1 = max(0, math.ceil(cx - S)), min(W - 1, math.floor(cx + S))\n            win = L[y0:y1 + 1, x0:x1 + 1]\n            D = np.sqrt(((win - [l, a, b]) ** 2).sum(-1) / c ** 2 + ((yy[y0:y1 + 1, x0:x1 + 1] - cy) ** 2 + (xx[y0:y1 + 1, x0:x1 + 1] - cx) ** 2) / S ** 2)\n            m = D < d[y0:y1 + 1, x0:x1 + 1]\n            d[y0:y1 + 1, x0:x1 + 1][m] = D[m]\n            lab[y0:y1 + 1, x0:x1 + 1][m] = k\n        for k in range(len(C)):\n            sel = lab.ravel() == k\n            if sel.any():\n                C[k] = F[sel].mean(0)\n    return lab\n\n\ndef compactness(lab):\n    vals = []\n    for k in np.unique(lab):\n        m = lab == k\n        per = (m[:, 1:] != m[:, :-1]).sum() + (m[1:] != m[:-1]).sum()\n        vals.append(4 * np.pi * m.sum() / max(per, 1) ** 2)\n    return float(np.mean(vals))\n\n\ndef check(ns, R):\n    L = color.rgb2lab(data.astronaut()[:160, 120:300])\n    for K, c in [(60, 10.0), (150, 5.0), (40, 30.0)]:\n        def t(K=K, c=c):\n            got = need(ns, \"slic\")(L, K, c)\n            want = slic_ref(L, K, c)\n            ensure(got.shape == want.shape and np.array_equal(got, want), f\"{int((got != want).sum())} piksel başvurudan farklı\")\n        R.test(f\"slic = pencereli başvuru (K = {K}, c = {c})\", t)\n    if \"slic\" not in ns:\n        return\n    rows = []\n    for c in (2.0, 10.0, 40.0):\n        lab = ns[\"slic\"](L, 100, c)\n        rgb = data.astronaut()[:160, 120:300].astype(float)\n        var = float(np.mean([rgb[lab == k].var(0).sum() for k in np.unique(lab)]))\n        rows.append([c, len(np.unique(lab)), f\"{compactness(lab):.3f}\", f\"{var:.0f}\"])\n    ensure(float(rows[2][2]) > float(rows[0][2]), \"c büyüdükçe süper pikseller kompaktlaşmalı\")\n    R.table(\"c'nin etkisi (K = 100)\", [\"c\", \"süper piksel\", \"ort. kompaktlık 4πA/P²\", \"ort. renk varyansı\"], rows)\n    t1 = bench(lambda: slic_ref(L, 100), repeat=1, number=1)\n    t2 = bench(lambda: ns[\"slic\"](L, 100), repeat=1, number=1)\n    R.table(\"160 × 180, K = 100, 10 iterasyon\", [\"yöntem\", \"süre\"], [[\"merkez başına pencere döngüsü (NumPy)\", fmt_time(t1)], [\"yoğun P × K maske (torch)\", fmt_time(t2)]],\n            note=\"Yoğun sürüm her piksel–merkez çiftini hesaplar (O(PK)); pencereli sürüm yalnız ~4S² pikseli (O(P)). GPU'da yoğun sürüm paralellikle kazanır; CPU'da pencere kazanır.\")\n",
 "x1": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import need, ensure\n\n\ndef check(ns, R):\n    rows = []\n    for name in (\"camera\", \"coins\", \"moon\", \"page\", \"text\"):\n        def t(name=name):\n            img = getattr(data, name)()\n            want, _ = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)\n            got = need(ns, \"otsu\")(img)\n            ensure(got == int(want), f\"{got}, OpenCV {int(want)}\")\n            rows.append([name, got])\n        R.test(f\"otsu = cv2 ({name})\", t)\n    if rows:\n        R.table(\"Otsu eşikleri\", [\"görüntü\", \"eşik\"], rows)\n\n    def ad():\n        img = data.page()\n        for b, C in [(11, 2), (25, 10), (5, 0), (15, -3)]:\n            ensure(np.array_equal(need(ns, \"adaptive_mean\")(img, b, C), cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, b, C)), f\"blok {b}, C {C}\")\n        t = ns[\"otsu\"](img)\n        g = (img > t).astype(np.uint8) * 255\n        return f\"'page' (eşit olmayan aydınlatma): Otsu sayfanın alt kısmını kaybeder; uyarlamalı eşik metni her yerde ayırır\"\n    R.test(\"adaptive_mean = cv2.adaptiveThreshold (MEAN_C)\", ad)\n",
 "x2": "import numpy as np\nfrom scipy import ndimage\nfrom skimage import data\nfrom labcheck import need, ensure\n\n\ndef check(ns, R):\n    img = data.coffee()[::2, ::2]\n\n    def sp():\n        lab, sp = need(ns, \"superpixel_kmeans\")(img, 300, 4)\n        ensure(lab.shape == img.shape[:2] and lab.max() < 4, \"k = 4 etiket\")\n        for s in np.unique(sp):\n            ensure(len(np.unique(lab[sp == s])) == 1, \"bir süper pikselin tüm pikselleri aynı etiketi almalı\")\n        pl = need(ns, \"pixel_kmeans\")(img, 4)\n        comps = lambda L: sum(ndimage.label(L == k)[1] for k in np.unique(L))\n        a, b = comps(lab), comps(pl)\n        ensure(a < b, f\"süper piksel tabanlı {a} bölge, piksel tabanlı {b}\")\n        R.table(\"coffee, k = 4\", [\"yöntem\", \"bağlı bölge sayısı\", \"kümelenen öğe\"], [[\"piksel k-means\", b, img.shape[0] * img.shape[1]], [\"süper piksel + k-means\", a, int(sp.max() + 1)]],\n                note=\"Süper pikseller hem kümelenecek öğe sayısını ~300 kat azaltır hem de tuz-biber benzeri tek piksellik bölgeleri ortadan kaldırır.\")\n    R.test(\"süper piksel + k-means\", sp)\n",
 "x3": "import numpy as np\nfrom scipy import ndimage\nfrom skimage import data\nfrom labcheck import need, ensure\n\n\ndef check(ns, R):\n    img = data.coins()\n\n    def rg():\n        seeds = [(20, 20), (110, 50), (110, 100), (20, 25)]\n        got = need(ns, \"region_grow\")(img, seeds, 25)\n        want = np.zeros(img.shape, np.int32)\n        for n, (y, x) in enumerate(seeds, 1):\n            if want[y, x]:\n                continue\n            ok = (np.abs(img.astype(int) - int(img[y, x])) <= 25) & (want == 0)\n            lab, _ = ndimage.label(ok, np.ones((3, 3)))\n            want[lab == lab[y, x]] = n\n        ensure(np.array_equal(got, want), \"başvuru ile farklı\")\n        return f\"bölge boyutları: {np.bincount(got.ravel())[1:].tolist()} (4. tohum 1. bölgenin içinde, atlanır)\"\n    R.test(\"region_grow\", rg)\n"
}

In [ ]:
import importlib, labcheck, checks_m9_11_segmentation as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Bölütleme:** görüntüyü anlamlı bölgelere ayırmak, yani her piksele bir etiket vermek. İki aile: **süreksizlik** (nokta, çizgi, kenar — M9.6) ve **benzerlik** (eşikleme, kümeleme, bölge büyütme, süper pikseller).

**Tekil nokta:** $|R(x,y)|\ge T$, $R$ = Laplacian yanıtı (8-komşulu çekirdek). **Çizgi:** Laplacian ince bir çizgide pozitif tepe ve iki yanında negatif tepeler verir; mutlak değer çizgiyi kalınlaştırır, bu yüzden **yalnız pozitif** kısım eşiklenir. Yön için dört çekirdek (yatay, dikey, iki çapraz); hangisinin $|Z_k|$'si en büyükse piksel o yöndeki çizgiyle "daha çok ilişkili"dir.

**k-means:** $\arg\min_C\sum_{i=1}^k\sum_{z\in C_i}\lVert z-m_i\rVert^2$ NP-zor; standart (Lloyd) algoritma: (1) başlangıç ortalamaları, (2) her örneği en yakın ortalamaya ata, (3) ortalamaları güncelle, (4) değişim $\le T$ olana dek tekrarla. Yakınsar ama global en iyiyi garanti etmez; sonuç başlangıca bağlıdır.

**GMM:** her küme bir Gauss $\mathcal N(\mu_k,\Sigma_k)$, karışım ağırlıkları $\pi_k$. **EM:** E adımı sorumluluklar $r_{nk}=\frac{\pi_k\mathcal N(z_n|\mu_k,\Sigma_k)}{\sum_j\pi_j\mathcal N(z_n|\mu_j,\Sigma_j)}$; M adımı $N_k=\sum_nr_{nk}$, $\pi_k=N_k/N$, $\mu_k=\frac1{N_k}\sum_nr_{nk}z_n$, $\Sigma_k=\frac1{N_k}\sum_nr_{nk}(z_n-\mu_k)(z_n-\mu_k)^T$. k-means, $\Sigma=\sigma^2I$ ve $\sigma\to0$ sınırıdır.

**SLIC (Achanta 2012):** $n_{sp}$ süper piksel için ızgara aralığı $s=\sqrt{n_{tp}/n_{sp}}$. (1) merkezleri ızgaraya koy ve $3\times3$ komşulukta en düşük gradyana kaydır, (2) her merkez yalnız çevresindeki $2s\times2s$ bölgedeki piksellere uzaklık hesaplar, $D_i(p)<d(p)$ ise etiketi günceller, (3) merkezleri ortalamalarla güncelle, (4) yakınsayana dek, (5) her bölgeyi ortalama rengiyle boya. Renk ve konum ölçekleri farklı olduğu için **normalize**: $D=\sqrt{(d_c/c)^2+(d_s/s)^2}$; küçük $c$ sınırlara yapışkan, büyük $c$ kompakt süper pikseller.

## 1 · Elle

<!-- çözüm -->

k-means 1. iterasyon: $m_1=1$'e yalnız 1 yakın; geri kalanların ortalaması $m_2=38/5=7.6$. 2. iterasyonda $\{1,2,3\}$ ve $\{10,11,12\}$ ayrılır: $m_1=2$, $m_2=11$; 3. iterasyonda değişmez.

GMM: $r=\frac{e^{-1/2}}{e^{-1/2}+e^{-9/2}}=\frac1{1+e^{-4}}$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Nokta ve çizgi**

- `pt`: Sıfırlar ortasında 10: $\begin{bmatrix}1&1&1\\1&-8&1\\1&1&1\end{bmatrix}$ ile $|R|$
- `ln`: Orta satırı 1 olan 3 × 3 yamada yatay çekirdek $\begin{bmatrix}-1&-1&-1\\2&2&2\\-1&-1&-1\end{bmatrix}$ yanıtı

**k-means, 1-B veri $\{1,2,3,10,11,12\}$, başlangıç $m_1=1$, $m_2=2$**

- `k1`: 1. iterasyondan sonra $m_2$
- `kf1`: Yakınsamada $m_1$
- `kf2`: Yakınsamada $m_2$

**SLIC ve GMM**

- `s`: 600 × 480 görüntü, 100 süper piksel: ızgara aralığı $s=\sqrt{n_{tp}/n_{sp}}$
- `D`: $d_c=20$, $c=10$, $d_s=30$, $s=50$: $D$
- `r`: 1-B GMM, eşit ağırlık, $\sigma=1$, ortalamalar 0 ve 4: $x=1$'in 1. bileşene sorumluluğu

In [ ]:
cevaplar = dict(
    pt=None,
    ln=None,
    k1=None,
    kf1=None,
    kf2=None,
    s=None,
    D=None,
    r=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"pt":[80,0],"ln":[6,0],"k1":[7.6,0],"kf1":[2,0],"kf2":[11,0],"s":[53.6656,0.001],"D":[2.08806,0.0001],"r":[0.982014,0.0001]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `point_detect(img, T)` — 8-komşulu Laplacian $|R|\ge T$ → 0/1; kenar satır/sütunları 0.
- `line_detect(img, kind, T)` — `kind`: `"h"` $\begin{bmatrix}-1&-1&-1\\2&2&2\\-1&-1&-1\end{bmatrix}$, `"v"` (devriği), `"d1"` (sol üst → sağ alt) $\begin{bmatrix}2&-1&-1\\-1&2&-1\\-1&-1&2\end{bmatrix}$, `"d2"` (sağ üst → sol alt, yatay aynası). Korelasyon; **pozitif** yanıt $\ge T$ → 1. Kenar 0.
- `kmeans(points, init, max_iter=300)` — `points`, `init`: liste listeleri; Lloyd; atamada eşitlikte küçük indis; boş küme ortalamasını korur; ortalamalar **hiç değişmeyince** dur. `(labels, means)` döndür.

In [ ]:
%%writefile stage_b.py
def point_detect(img, T):
    ...


def line_detect(img, kind, T):
    ...


def kmeans(points, init, max_iter=300):
    ...


print(kmeans([[1], [2], [3], [10], [11], [12]], [[1], [2]]))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok** — yalnız iterasyon döngüsü (`for`/`while` bir tane) serbest.

- `kmeans_np(Z, init, max_iter=300)` — `(N, d)` `float64`; `sklearn.cluster.KMeans(init=init, n_init=1, tol=0, algorithm="lloyd")` ile aynı etiketler ve merkezler. İpucu: uzaklıklar `((Z[:, None] - m) ** 2).sum(-1)`, yeni ortalamalar `np.add.at` ya da `np.bincount(labels, weights=…)` ile.
- `quantize(img, k, seed=0)` — RGB `uint8`: `rng = np.random.default_rng(seed)`, başlangıç `Z[rng.choice(N, k, replace=False)]`; `kmeans_np`; her pikseli merkez rengine boya (`np.rint`, `uint8`). `(out, labels (H, W))`.

In [ ]:
%%writefile stage_c.py
import numpy as np


def kmeans_np(Z, init, max_iter=300):
    ...


def quantize(img, k, seed=0):
    ...


from skimage import data
out, lab = quantize(data.astronaut()[::4, ::4], 8)
print(out.shape, lab.max())

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

`gmm_em(Z, mu0, iters=50, reg=1e-6)` — `Z (N, d)` `float64` tensör, `mu0 (K, d)`. Başlangıç: $\pi_k=1/K$, $\Sigma_k=\mathrm{cov}(Z)+\text{reg}\cdot I$ (yansız). `iters` kez E + M (M'de $\Sigma_k$'ye `reg·I` ekle). Log-olasılıkları Cholesky ile ($\log|\Sigma|=2\sum\log\mathrm{diag}L$) ve `logsumexp` ile hesapla. `(mu, cov, w, mean_loglik)` — son değer **son E adımındaki** ortalama log-olabilirlik.

Kontrol, sonucu `GaussianMixture(weights_init, means_init, precisions_init, tol=0, max_iter=iters)` ile karşılaştırır ve uzamış (eliptik) kümelerde GMM'in k-means'i nasıl geçtiğini gösterir.

In [ ]:
%%writefile stage_d.py
import math
import torch


def gmm_em(Z, mu0, iters=50, reg=1e-6):
    ...


Z = torch.randn(200, 2, dtype=torch.float64)
print(gmm_em(Z, Z[:3])[3])

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`slic(lab, K, c=10.0, iters=10)` — `lab (H, W, 3)` `float64` NumPy (L*a*b*); `(H, W)` `int64` etiket. Tanım (kontrol aynı tanımı döngülerle uygular, **birebir** eşleşmeli):

1. $S$ = `round(sqrt(H·W/K))`. Izgara: `y ∈ range(S//2, H, S)`, `x ∈ range(S//2, W, S)` (satır sırası).
2. Gradyan $G=\lVert I(y,x+1)-I(y,x-1)\rVert^2+\lVert I(y+1,x)-I(y-1,x)\rVert^2$ (`edge` dolgu); merkezi $3\times3$'te (görüntüye kırpılmış, satır sırası, ilk en küçük) en düşük $G$'ye taşı. Merkez $[L,a,b,y,x]$.
3. Her iterasyonda: piksel $p$ için adaylar $|y_p-c_y|\le S$ ve $|x_p-c_x|\le S$ olan merkezler; $D=\sqrt{d_c^2/c^2+d_s^2/S^2}$; etiket = en küçük $D$ (eşitlikte küçük indis); adayı olmayan piksel eski etiketini korur (başta −1).
4. Merkez = atanmış piksellerin $[L,a,b,y,x]$ ortalaması (boşsa eski).

Merkez hesabı 2. adımda döngülü olabilir; 3–4. adımlar tensör işlemleri (`index_add_`) ile, yalnız iterasyon döngüsü.

In [ ]:
%%writefile stage_e.py
import math
import numpy as np
import torch


def slic(lab, K, c=10.0, iters=10):
    ...


from skimage import data, color
L = color.rgb2lab(data.astronaut()[:64, :64])
print(np.unique(slic(L, 16)).size)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Eşikleme: Otsu ve uyarlamalı ortalama (cv2 ile birebir)

- `otsu(img)` — `cv2.threshold(img, 0, 255, THRESH_BINARY + THRESH_OTSU)`'nun eşiği: $\sigma_B^2(k)=\frac{(m_GP_1(k)-m(k))^2}{P_1(k)(1-P_1(k))}$'yi en büyük yapan $k$ (ilk). Histogramdan, döngüsüz.
- `adaptive_mean(img, block, C)` — `cv2.adaptiveThreshold(img, 255, ADAPTIVE_THRESH_MEAN_C, THRESH_BINARY, block, C)` ile aynı: ortalama `cv2.boxFilter(img, -1, (block, block), borderType=cv2.BORDER_REPLICATE)` (uint8, yuvarlanmış), `img > mean − C` → 255.

In [ ]:
%%writefile extra_x1.py
import numpy as np
import cv2


def otsu(img):
    ...


def adaptive_mean(img, block, C):
    ...


from skimage import data
print(otsu(data.camera()))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Süper piksel + k-means bölütleme (alıştırma 6.1)

Alıştırma 6.1: süper pikseller üzerinde k-means.

- `superpixel_kmeans(img, n_segments, k, seed=0)` — `skimage.segmentation.slic(img, n_segments=n_segments, compactness=10, start_label=0)`; her süper pikselin ortalama RGB'si; bu ortalamalar üzerinde k-means (`rng.choice` ile başlangıç, ilk alıştırmadaki `kmeans_np` mantığı ya da `sklearn.cluster.KMeans(init=…, n_init=1)`); her pikselin etiketi = süper pikselinin küme etiketi. `(labels, sp)`.
- `pixel_kmeans(img, k, seed=0)` — karşılaştırma için piksel düzeyinde aynı k-means.

In [ ]:
%%writefile extra_x2.py
import numpy as np
from skimage.segmentation import slic
from sklearn.cluster import KMeans


def superpixel_kmeans(img, n_segments, k, seed=0):
    ...


def pixel_kmeans(img, k, seed=0):
    ...


from skimage import data
lab, sp = superpixel_kmeans(data.coffee()[::2, ::2], 300, 4)
print(lab.max(), sp.max())

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Tohumlu bölge büyütme

`region_grow(img, seeds, T)` — tohum listesi `[(y, x), …]`; her tohumdan 8-bağlantıyla, $|f(p)-f(\text{tohum})|\le T$ koşulunu sağlayan piksellere genişle (BFS); etiket haritası `int32`, tohum sırasına göre 1, 2, …; bir piksel daha önce başka bir tohumun bölgesine girdiyse değişmez (ilk gelen kazanır, tohumlar sırayla işlenir).

In [ ]:
%%writefile extra_x3.py
import numpy as np
from collections import deque


def region_grow(img, seeds, T):
    ...


from skimage import data
print(np.bincount(region_grow(data.coins(), [(100, 100), (50, 300)], 20).ravel()))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Superpixel Sampling Networks: türevlenebilir SLIC

[Jampani, Sun, Liu, Yang, Kautz — Superpixel Sampling Networks (ECCV 2018)](https://arxiv.org/abs/1807.10174)

Bu modülde yazdığın SLIC'in sert atamasını, piksel–süper piksel benzerliklerinin softmax'ıyla (yumuşak atama) değiştirerek türevlenebilir hale getir; bir CNN'in ürettiği özniteliklerle uçtan uca eğit (SSN). BSDS500'de sınır yakalama (boundary recall) ve yetersiz bölütleme hatasını (undersegmentation error) SLIC ile karşılaştır.


**Adımlar**

1. Yumuşak atama $Q_{pi}=e^{-\lVert F_p-S_i\rVert^2}$ (yalnız çevredeki 9 süper piksel), merkez güncelleme $S_i=\frac{\sum_pQ_{pi}F_p}{\sum_pQ_{pi}}$.
2. Öznitelik ağı: birkaç konvolüsyon + konum özniteliği (ölçekli x, y).
3. Kayıp: etiketlerin süper piksellere izdüşümü ve geri izdüşümü üzerinden yeniden kurma + kompaktlık.
4. Görev özel: segmentasyon etiketleri yerine optik akış ya da derinlik ile eğitme denemesi.

**Kabul ölçütleri**

- Aynı süper piksel sayısında SSN'nin sınır yakalaması SLIC'ten yüksek (grafik: sayı 100–1000).
- Yumuşak atamanın sıcaklığı (ölçek) için ablasyon.
- Bu modüldeki e aşamasının sert SLIC'iyle SSN'nin çıkarım süresi karşılaştırması.

## Kaynaklar

- [Gonzalez & Woods — Digital Image Processing, 4. baskı, Bölüm 10.2, 10.3, 10.5](https://www.imageprocessingplace.com/) — ©
- [Achanta ve ark. — SLIC Superpixels Compared to State-of-the-Art Superpixel Methods (TPAMI 2012)](https://doi.org/10.1109/TPAMI.2012.120) — ©
- [scikit-learn — KMeans, GaussianMixture](https://scikit-learn.org/stable/modules/clustering.html) — BSD-3
- [Otsu — A Threshold Selection Method from Gray-Level Histograms (1979)](https://doi.org/10.1109/TSMC.1979.4310076) — ©
- [scikit-image — segmentation.slic](https://scikit-image.org/docs/stable/api/skimage.segmentation.html) — BSD-3